# 02 - Prototipo Monte Carlo (sin Spark) leyendo desde MinIO

Este notebook corre en el contenedor de **JupyterLab**, usa Python puro (`pandas`, `numpy`) y lee los datos directamente desde **MinIO**.

Para simplificar:
- Reutilizamos el dataset de features en Gold: `gold/smart_inventory/dataset_features`.
- Usamos la columna `target_ventas_proximo_dia` como proxy de `demanda_esperada`.
- Implementamos una simulación Monte Carlo pequeña (pocos productos y fechas) para entender el comportamiento.


In [ ]:
%pip install minio pandas pyarrow numpy matplotlib --quiet
import io
import numpy as np
import pandas as pd
from minio import Minio
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

In [ ]:
# Conexión a MinIO (mismo patrón que en el notebook 01)
MINIO_ENDPOINT = "minio:9000"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "admin123"
MINIO_SECURE = False

client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=MINIO_SECURE,
)

GOLD_BUCKET = "gold"
GOLD_PREFIX_FEATURES = "smart_inventory/dataset_features"

print("Conectado a MinIO en:", MINIO_ENDPOINT)

In [ ]:
# Helper para leer algunos archivos Parquet desde MinIO
def leer_parquet_minio(bucket: str, prefix: str, max_files: int = 5) -> pd.DataFrame:
    objetos = client.list_objects(bucket, prefix=prefix, recursive=True)
    dfs = []
    for obj in objetos:
        if not obj.object_name.endswith(".parquet"):
            continue
        print(f"Leyendo {bucket}/{obj.object_name} ...")
        data = client.get_object(bucket, obj.object_name).read()
        df = pd.read_parquet(io.BytesIO(data))
        dfs.append(df)
        if len(dfs) >= max_files:
            break
    if not dfs:
        raise ValueError(f"No se encontraron Parquet en {bucket}/{prefix}")
    return pd.concat(dfs, ignore_index=True)

df_gold = leer_parquet_minio(GOLD_BUCKET, GOLD_PREFIX_FEATURES, max_files=5)
print("Filas en la muestra de Gold:", len(df_gold))
display(df_gold.head())

## 1. Preparamos la base para simulación

Tomamos un subconjunto pequeño de productos y fechas para que la simulación sea ligera.

Como proxy de demanda esperada usamos `target_ventas_proximo_dia`.


In [ ]:
# Aseguramos tipos correctos
if "fecha" in df_gold.columns:
    df_gold["fecha"] = pd.to_datetime(df_gold["fecha"])

# Filtramos filas con target no nulo
df_base = df_gold.dropna(subset=["target_ventas_proximo_dia"]).copy()

# Elegimos hasta 2 productos de muestra
productos = df_base["producto_id"].dropna().unique().tolist()
productos_muestra = productos[:2]
print("Productos muestra:", productos_muestra)

df_sample = (
    df_base[df_base["producto_id"].isin(productos_muestra)]
    .sort_values(["producto_id", "fecha"])
    .loc[:, ["producto_id", "fecha", "target_ventas_proximo_dia"]]
)

# Limitamos cantidad de filas para que sea rápido
        
df_sample = df_sample.head(200).reset_index(drop=True)

df_sample.rename(columns={"target_ventas_proximo_dia": "demanda_esperada"}, inplace=True)
display(df_sample.head())
print("Filas en la muestra para simulación:", len(df_sample))

## 2. Parámetros de simulación Monte Carlo

Definimos:
- `NUM_SIMULACIONES`: escenarios por combinación (`producto_id`, `fecha`).
- `VOLATILIDAD_PCT`: desvío estándar relativo respecto a la demanda esperada.
- `STOCK_INICIAL`: stock fijo para evaluar riesgo de quiebre.


In [ ]:
NUM_SIMULACIONES = 100
VOLATILIDAD_PCT = 0.2   # 20% de la demanda esperada
STOCK_INICIAL = 500
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

print("Simulaciones:", NUM_SIMULACIONES)
print("Volatilidad (%):", VOLATILIDAD_PCT * 100)
print("Stock inicial:", STOCK_INICIAL)

## 3. Generación de escenarios aleatorios

Para cada fila (`producto_id`, `fecha`, `demanda_esperada`):
- Generamos `NUM_SIMULACIONES` demandas simuladas ~ Normal(μ, σ), con `μ = demanda_esperada`, `σ = VOLATILIDAD_PCT * μ`.
- Cortamos valores negativos a 0.

Luego armamos un DataFrame con columnas:
- `producto_id`, `fecha`, `simulacion_id`, `demanda_simulada`.


In [ ]:
rows = []
for idx, row in df_sample.iterrows():
    mu = float(row["demanda_esperada"])
    sigma = max(mu * VOLATILIDAD_PCT, 1e-6)  # evitar sigma=0
    draws = np.random.normal(loc=mu, scale=sigma, size=NUM_SIMULACIONES)
    draws = np.clip(draws, 0, None)  # no permitir demanda negativa
    for sim_id, d in enumerate(draws):
        rows.append({
            "producto_id": row["producto_id"],
            "fecha": row["fecha"],
            "simulacion_id": sim_id,
            "demanda_simulada": d,
        })

df_mc = pd.DataFrame(rows)
print("Filas en df_mc:", len(df_mc))
display(df_mc.head())

## 4. Cálculo de stock proyectado y quiebre

Para cada combinación (`producto_id`, `simulacion_id`) ordenada por fecha:
- Calculamos `demanda_acumulada`.
- Calculamos `stock_proyectado = STOCK_INICIAL - demanda_acumulada`.
- Definimos `quiebre_stock = 1` si `stock_proyectado < 0`, si no `0`.


In [ ]:
df_mc_sorted = df_mc.sort_values(["producto_id", "simulacion_id", "fecha"]).copy()

df_mc_sorted["demanda_acumulada"] = (
    df_mc_sorted
    .groupby(["producto_id", "simulacion_id"])["demanda_simulada"]
    .cumsum()
)

df_mc_sorted["stock_proyectado"] = STOCK_INICIAL - df_mc_sorted["demanda_acumulada"]
df_mc_sorted["quiebre_stock"] = (df_mc_sorted["stock_proyectado"] < 0).astype(int)

display(df_mc_sorted.head())

## 5. Agregación de resultados: probabilidad de quiebre y percentiles

Agregamos a nivel (`producto_id`, `fecha`):
- `probabilidad_quiebre` = promedio de `quiebre_stock`.
- `demanda_p50` = percentil 50 de `demanda_simulada`.
- `demanda_p95` = percentil 95 de `demanda_simulada`.
- `stock_promedio_esperado` = promedio de `stock_proyectado`.


In [ ]:
def p50(x):
    return np.percentile(x, 50)

def p95(x):
    return np.percentile(x, 95)

agg = (
    df_mc_sorted
    .groupby(["producto_id", "fecha"], as_index=False)
    .agg(
        probabilidad_quiebre=("quiebre_stock", "mean"),
        demanda_p50=("demanda_simulada", p50),
        demanda_p95_critica=("demanda_simulada", p95),
        stock_promedio_esperado=("stock_proyectado", "mean"),
    )
    .sort_values(["producto_id", "fecha"])
)

        
display(agg.head(30))

## 6. Visualización para un producto de ejemplo

Graficamos la evolución de:
- Probabilidad de quiebre de stock.
- Stock promedio esperado.


In [ ]:
producto_demo = agg["producto_id"].unique()[0]
print("Producto demo:", producto_demo)

df_demo = agg[agg["producto_id"] == producto_demo].copy()
df_demo = df_demo.sort_values("fecha")

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.set_xlabel("Fecha")
ax1.set_ylabel("Probabilidad de quiebre", color="tab:red")
ax1.plot(df_demo["fecha"], df_demo["probabilidad_quiebre"], color="tab:red", marker="o")
ax1.tick_params(axis="y", labelcolor="tab:red")
plt.xticks(rotation=45)

ax2 = ax1.twinx()
ax2.set_ylabel("Stock promedio esperado", color="tab:blue")
ax2.plot(df_demo["fecha"], df_demo["stock_promedio_esperado"], color="tab:blue", marker="x")
ax2.tick_params(axis="y", labelcolor="tab:blue")

fig.tight_layout()
plt.title(f"Prob. quiebre y stock promedio - {producto_demo}")
plt.show()